# Download các thư viện cần thiết

In [1]:
import os
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Tạo hàm để đọc file parquet (đọc các file parquet - data sau khi được processed)

In [2]:
def read_parquet(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    user_chunk_files = [file for file in files if 'user_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    user_chunk_df = pl.concat([pl.read_parquet(file) for file in user_chunk_files]) if user_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return user_chunk_df

In [3]:
def read_parquet_item(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    item_chunk_files = [file for file in files if 'item_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    item_chunk_df = pl.concat([pl.read_parquet(file) for file in item_chunk_files]) if item_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return item_chunk_df

In [4]:
def read_parquet_purchase(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    purchase_chunk_files = [file for file in files if 'purchase_history_daily_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    purchase_chunk_df = pl.concat([pl.read_parquet(file) for file in purchase_chunk_files]) if purchase_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return purchase_chunk_df

# Tạo hàm lưu file parquet sau mỗi task

In [5]:
def split_and_save_parquet(df, num_files, output_dir):
    """
    Tách DataFrame thành nhiều file Parquet và lưu vào thư mục đích.
    
    :param df: DataFrame cần tách
    :param num_files: Số lượng file Parquet muốn tách
    :param output_dir: Thư mục lưu các file Parquet
    """
    # Đảm bảo thư mục tồn tại
    os.makedirs(output_dir, exist_ok=True)
    
    # Tính số dòng mỗi file sẽ có
    num_rows = df.height
    rows_per_file = num_rows // num_files

    # Tách DataFrame thành các phần và lưu mỗi phần vào một file Parquet
    for i in range(num_files):
        start_row = i * rows_per_file
        # Đảm bảo phần cuối cùng sẽ chứa tất cả các dòng còn lại
        end_row = (i + 1) * rows_per_file if i < num_files - 1 else num_rows
        
        # Tách phần DataFrame
        split_df = df[start_row:end_row]
        
        # Lưu phần DataFrame vào file .parquet
        file_path = os.path.join(output_dir, f"sale_pers.purchase_history_daily_chunk_{i}.parquet")
        split_df.write_parquet(file_path)
        print(f"Đã lưu file: {file_path}")

# Load dataset ban đầu

In [6]:
purchase = read_parquet_purchase(".././dataset")
purchase.head()

timestamp,user_id,item_id,event_type,event_value,price,date_key,quantity,customer_id,created_date,updated_date,channel,payment,location,discount,is_deleted
i64,str,str,str,"decimal[38,4]","decimal[38,4]",i32,i32,i32,datetime[μs],datetime[μs],str,str,i32,"decimal[38,4]",bool
1705655565,"""dca9b07f023d82f2cd88759718b817…","""4684000000001""","""Purchase""",1.0000,350000.0000,20240119,1,6602502,2024-01-19 09:12:45.237,2024-01-19 09:12:45.237,"""In-Store""","""Cà thẻ""",889,0.0000,false
1705225247,"""1987171b8293efb6d96097f65a4c67…","""1618000000001""","""Purchase""",1.0000,205000.0000,20240114,1,692908,2024-01-14 09:40:47.600,2024-01-14 09:40:47.600,"""In-Store""","""Tiền mặt""",150,0.0000,false
1705255322,"""bd86a3dc8e396c963088fb9ecb5d5b…","""0029010030006""","""Purchase""",1.0000,69000.0000,20240114,1,6323735,2024-01-14 18:02:02.450,2024-01-14 18:02:02.450,"""In-Store""","""Tiền mặt""",120,0.0000,false
1705671047,"""5701599ade672af42b94e87aafc7b9…","""0206024330001""","""Purchase""",1.0000,279000.0000,20240119,1,6569899,2024-01-19 13:30:47.610,2024-01-19 13:30:47.610,"""In-Store""","""VietQR""",361,0.0000,false
1705256022,"""aa55cd0ca4060b7144e883e1a5bedf…","""2123004000001""","""Purchase""",1.0000,189000.0000,20240114,1,6720829,2024-01-14 18:13:42.837,2024-01-14 18:13:42.837,"""In-Store""","""VietQR""",386,0.0000,false


In [7]:
item = read_parquet_item(".././dataset")
item.head()

p_id,item_id,price,category_l1_id,category_l1,category_l2_id,category_l2,category_l3_id,category_l3,category_id,category,description,brand,manufacturer,creation_timestamp,is_deleted,created_date,updated_date,sync_status_id,last_sync_date,sync_error_message,image_url,gender_target,age_group,item_type,gp,weight,color,size,origin,volume,material,sale_status,description_new
i32,str,"decimal[38,4]",i32,str,i32,str,i32,str,i32,str,str,str,str,i64,bool,datetime[μs],datetime[μs],i32,datetime[μs],str,str,str,str,str,"decimal[38,4]",f32,str,str,str,str,str,i32,str
17065,"""0502020000004""",99000.0000,1,"""Babycare""",35,"""Bình sữa, phụ kiện""",7050,"""Núm ty""",7058,"""Núm ty Dr Brown""","""Không xác định""","""Dr.Brown's""","""Không xác định""",1333531544,false,2012-04-04 09:25:44.240,2025-08-18 09:59:19.847,2,2025-07-18 17:59:29.898256,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",36828.0000,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",0,"""Chi tiết sản phẩm …"
72370,"""0010290040150""",69000.0000,3292,"""Thời trang""",3958,"""Cơ cấu hàng cũ""",7007,"""Thời trang bé trai, bé gái cũ""",6987,"""Bộ quần áo bé gái""","""Không xác định""","""Con Cưng""","""Không xác định""",1503046250,false,2017-08-18 08:50:50.713,2025-09-18 16:05:42.360,null,null,null,"""Không xác định""","""Bé Gái""","""Từ 3Y""","""Bộ quần áo""",0.0000,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",0,"""Không xác định"""
31154,"""0008010000015""",45000.0000,351,"""Đồ chơi & Sách""",2033,"""0-1Y""",2118,"""Gặm nướu""",2121,"""Gặm nướu khác""","""- Chất liệu: Sản phẩm được làm…","""Thương hiệu khác""","""Không xác định""",1358501584,false,2013-01-18 09:33:04.260,2025-09-27 00:05:36.233,2,2025-07-18 17:59:29.898256,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",14490.0000,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",0,"""Chi tiết sản phẩm …"
46123,"""0020010000094""",401000.0000,2222,"""Tã""",2272,"""Merries""",2275,"""Merries""",2276,"""Merries_Sơ Sinh""","""﻿﻿Tã dán Merries size S 82 miế…","""Merries Nhật""","""Không xác định""",1400062039,false,2014-05-14 10:07:19.603,2025-09-27 00:05:36.233,2,2025-07-18 17:59:29.898256,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",59749.0000,null,"""Không xác định""","""Không xác định""","""Nhật Bản, Nhật Bản""","""Không xác định""","""Giấy, bột giấy, vải không dệt,…",0,"""Không xác định"""
46127,"""0020010000098""",401000.0000,2222,"""Tã""",2272,"""Merries""",2275,"""Merries""",2278,"""Merries_Tã Quần""","""﻿﻿﻿Bỉm tã quần Merries size M …","""Merries Nhật""","""Không xác định""",1400062040,false,2014-05-14 10:07:20.370,2025-09-27 00:05:36.233,2,2025-07-18 17:59:29.898256,null,"""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",65764.0000,null,"""Không xác định""","""Không xác định""","""Nhật Bản, Nhật Bản""","""Không xác định""","""Giấy, bột giấy, vải không dệt,…",0,"""Không xác định"""


In [8]:
user = read_parquet(".././dataset")
user.head()

customer_id,gender,location,province,membership,timestamp,created_date,updated_date,sync_status_id,last_sync_date,sync_error_message,region,location_name,install_app,install_date,district,user_id,is_deleted
i32,str,i32,str,str,i64,datetime[μs],datetime[μs],i32,datetime[μs],str,str,str,str,i64,str,str,bool
2321395,"""Nữ""",429,"""Hồ Chí Minh""","""Standard""",1557247995,2019-05-07 16:53:15.147,2025-07-07 15:33:10.201316,2,2025-07-16 11:56:29.751170,null,"""Đông Nam Bộ""","""HCM - 2 Đường Số 40""","""In-Store""",1557187200,"""Bình Tân""","""94508bee1b3242a0248c75c9a86681…",false
2321413,"""Nữ""",170,"""Đồng Tháp""","""Standard""",1557248431,2019-05-07 17:00:31.273,2025-07-07 15:33:10.201316,2,2025-07-16 11:56:29.751170,null,"""Đồng bằng sông Cửu Long""","""ĐTH - 281-283 Nguyễn Sinh Sắc""","""In-Store""",1629244800,"""Sa Đéc""","""9029189445ed6a523d397692174d36…",false
2321355,"""Nữ""",144,"""Khánh Hòa""","""Gold""",1557247196,2019-05-07 16:39:56.703,2025-09-23 08:08:10.457,2,2025-07-16 11:56:29.751170,null,"""Duyên hải Nam Trung Bộ""","""KHO - 2193 -2195 - 2197 Đại Lộ…","""In-Store""",1735948800,"""Cam Ranh""","""e5952cc17349bfc6a6c98137e81e82…",false
2321433,"""Nữ""",333,"""Quảng Trị""","""Standard""",1557248908,2019-05-07 17:08:28.650,2025-07-07 15:33:10.201316,2,2025-07-16 11:56:29.751170,null,"""Bắc Trung Bộ""","""QTR - 84 Quốc Lộ 9""","""In-Store""",1557187200,"""Đông Hà""","""c649dcf91aeaae3a46637800c45df0…",false
2321313,"""Nam""",719,"""Lâm Đồng""","""Gold""",1557246359,2019-05-07 16:25:59.400,2025-09-16 19:51:44.287,2,2025-07-16 11:56:29.751170,null,"""Tây Nguyên""","""LDO - Liên Trung""","""In-Store""",1620259200,"""Lâm Hà""","""1f99dc15d160ff015f9807caeaadc6…",false


# Load các dataframe cần thiết

In [11]:
purchase_df = read_parquet_purchase(".././preprocessed-feature")
purchase_df.head()

item_id,quantity,customer_id,created_date,location,price,log_price,discount_rate,channel,payment_bucket,time_between_purchases,month,seasonal_trend,product_engagement_level,avg_transaction_amount_per_purchase,segment_name,avg_cat_l1_per_purchase,segment_name_right
str,i32,i32,datetime[μs],i32,f64,f64,f64,str,str,duration[μs],i8,str,str,f64,str,f64,str
"""2803000000013""",2,6604637,2024-10-14 18:02:15.860,578,44100.0,10.694238,0.1,"""In-Store""","""card""",360d 22h 52m 10s 780ms,10,"""Autumn""","""High""",580106.153069,"""Trung cấp""",1.686047,"""Mua vừa"""
"""0020020000172""",1,6594820,2024-10-14 07:37:39.637,831,30000.0,10.308986,0.0,"""In-Store""","""cash""",352d 10h 35m 46s 53ms,10,"""Autumn""","""High""",274824.47673,"""Bình dân""",1.895522,"""Mua vừa"""
"""0020120000014""",1,7957733,2024-10-14 14:50:28.070,63,80500.0,11.296025,0.3,"""In-Store""","""cash""",0µs,10,"""Autumn""","""High""",495850.0,"""Trung cấp""",4.0,"""Mua nhiều"""
"""1771000000002""",1,7559596,2024-10-14 12:13:40.113,155,45000.0,10.71444,0.0,"""In-Store""","""cash""",136d 22h 8m 54s 46ms,10,"""Autumn""","""High""",352361.478357,"""Bình dân""",1.222222,"""Mua ít"""
"""1308000000003""",2,6885875,2024-10-14 16:20:13.623,544,75000.0,11.225257,0.0,"""In-Store""","""card""",257d 20h 51m 1s 654ms,10,"""Autumn""","""High""",384639.619569,"""Bình dân""",1.571429,"""Mua vừa"""


In [ ]:
# purchase_df = purchase_df.drop('top10_co_items_right')

In [12]:
item_df = read_parquet_item(".././preprocessed-feature")
item_df.head()

item_id,price,category_l1,category,brand_final,target_user_group_final,item_type_final,sale_status,age_bucket_final,price_norm
str,"decimal[38,4]",str,str,str,str,str,i32,str,f64
"""0502020000004""",99000.0000,"""babycare""","""núm ty dr brown""","""dr.brown's""","""sơ sinh""",null,0,"""0-6m""",-0.178933
"""0010290040150""",69000.0000,"""thời trang""","""bộ quần áo bé gái""","""con cưng""","""bé gái""","""bộ quần áo""",0,"""2-4y""",-0.237627
"""0008010000015""",45000.0000,"""đồ chơi & sách""","""gặm nướu khác""","""thương hiệu khác""","""bé trai""",null,0,"""1-4y""",-0.284582
"""0020010000094""",401000.0000,"""tã""","""merries_sơ sinh""","""merries nhật""","""sơ sinh""",null,0,"""0-6m""",0.411922
"""0020010000098""",401000.0000,"""tã""","""merries_tã quần""","""merries nhật""","""sơ sinh""",null,0,"""6-12m""",0.411922


In [13]:
user_df = read_parquet(".././preprocessed-feature")
user_df.head()

customer_id,gender,location,province,membership,region,location_name,install_app,district,milk_segment_preference,diaper_segment_preference
i32,str,i32,str,str,str,str,str,str,i64,i64
2102259,"""Nữ""",838,"""Kiên Giang""","""Standard""","""Đồng bằng sông Cửu Long""","""KGI - Lô L10-12 QL61""","""In-Store""","""Gò Quao""",null,null
2102190,"""Nữ""",728,"""Gia Lai""","""Standard""","""Tây Nguyên""","""GLA - 113 Hai Bà Trưng""","""In-Store""","""Pleiku""",null,2
2102266,"""Nữ""",535,"""Đồng Nai""","""Standard""","""Đông Nam Bộ""","""DON - 537 Cách Mạng Tháng Tám""","""In-Store""","""Biên Hòa""",null,null
2102268,"""Nam""",545,"""Hồ Chí Minh""","""Standard""","""Đông Nam Bộ""","""HCM - 304B Trường Chinh""","""In-Store""","""Tân Bình""",null,null
2102273,"""Nữ""",365,"""Hồ Chí Minh""","""Standard""","""Đông Nam Bộ""","""HCM - 266A Tỉnh Lộ 15""","""In-Store""","""Củ Chi""",null,null


# XÂY DỰNG BẢNG FEATURE - LABEL

Sắp xếp theo thời gian

In [14]:
purchase_df = purchase_df.sort("created_date")  # mặc định tăng dần theo thời gian
purchase_df.head()

item_id,quantity,customer_id,created_date,location,price,log_price,discount_rate,channel,payment_bucket,time_between_purchases,month,seasonal_trend,product_engagement_level,avg_transaction_amount_per_purchase,segment_name,avg_cat_l1_per_purchase,segment_name_right
str,i32,i32,datetime[μs],i32,f64,f64,f64,str,str,duration[μs],i8,str,str,f64,str,f64,str
"""2006000000006""",1,4689434,2024-01-01 06:44:59.037,627,35200.0,10.46883,0.451713,"""In-Store""","""cash""",365d 8h 31m 6s 243ms,1,"""Winter""","""High""",1.1825e6,"""Trung cấp""",1.378641,"""Mua ít"""
"""0020010000440""",1,5279260,2024-01-01 06:48:28.537,547,465000.0,13.049795,0.0,"""In-Store""","""cash""",0µs,1,"""Winter""","""High""",465000.0,"""Trung cấp""",1.0,"""Mua ít"""
"""2485000000004""",1,4190229,2024-01-01 06:49:32.443,483,469000.0,13.05836,0.0,"""In-Store""","""cash""",364d 8h 53m 19s 84ms,1,"""Winter""","""High""",511970.68006,"""Trung cấp""",1.162791,"""Mua ít"""
"""2482000000004""",1,6530105,2024-01-01 06:51:11.120,348,525000.0,13.171155,0.0,"""In-Store""","""cash""",360d 7h 56m 39s 227ms,1,"""Winter""","""High""",339501.965496,"""Bình dân""",1.631579,"""Mua vừa"""
"""6767000000003""",1,6393411,2024-01-01 06:52:48.570,560,265000.0,12.487489,0.070175,"""In-Store""","""cash""",301d 12h 1m 4s 490ms,1,"""Winter""","""High""",386749.068627,"""Bình dân""",1.705882,"""Mua vừa"""


In [15]:
import polars as pl
from datetime import datetime

def build_feature_label(
    transactions_lf: pl.LazyFrame,  # purchase_df.lazy()
    items_lf: pl.LazyFrame,         # item_df.lazy()
    users_lf: pl.LazyFrame,         # chưa dùng, để đúng spec
    begin_hist: datetime,
    end_hist: datetime,
    begin_recent: datetime,
    end_recent: datetime,
) -> pl.LazyFrame:
    """
    Tạo bảng Feature - Label cho bài toán khuyến nghị top-k.

    Output columns:
        - customer_id
        - item_id
        - brand_counts
        - age_counts
        - category_counts
        - segment_counts            (# lần mua cùng (category_l1, segment_name) với item hiện tại)
        - target_user_group_counts
        - time_since_last_purchase_in_B_category
        - Y
    """

    # 0. Làm sạch transactions_lf: bỏ cột segment_name_right nếu tồn tại (do join trước đó sinh ra)
    tx_cols = transactions_lf.columns
    if "segment_name_right" in tx_cols:
        base_tx = transactions_lf.drop("segment_name_right")
    else:
        base_tx = transactions_lf

    # 1. Filter giai đoạn HIST và RECENT trên bảng giao dịch
    hist_lf = (
        base_tx
        .filter(
            pl.col("created_date").is_between(begin_hist, end_hist, closed="both")
        )
    )

    recent_lf = (
        base_tx
        .filter(
            pl.col("created_date").is_between(begin_recent, end_recent, closed="both")
        )
    )

    # 2. Thuộc tính item cơ bản (từ item_df)
    item_attrs = items_lf.select([
        "item_id",
        "brand_final",
        "age_bucket_final",
        "category",
        "category_l1",
        "target_user_group_final",
    ])

    # 3. Mapping item_id -> segment_name đại diện (mode trong HIST)
    item_segment = (
        hist_lf
        .group_by("item_id")
        .agg(pl.col("segment_name").mode().alias("segment_name_list"))
        .with_columns(
            pl.col("segment_name_list").list.first().alias("segment_name")
        )
        .select(["item_id", "segment_name"])
    )

    # 4. HIST đã gắn đầy đủ: brand/age/category/category_l1/segment_name
    hist_enriched = (
        hist_lf
        .join(item_attrs,   on="item_id", how="left")
        .join(item_segment, on="item_id", how="left")
    )

    # 5. Feature 1: brand_counts
    brand_counts = (
        hist_enriched
        .group_by(["customer_id", "brand_final"])
        .agg(pl.len().alias("brand_counts"))
    )

    # 6. Feature 2: age_counts
    age_counts = (
        hist_enriched
        .group_by(["customer_id", "age_bucket_final"])
        .agg(pl.len().alias("age_counts"))
    )

    # 7. Feature 3: category_counts
    category_counts = (
        hist_enriched
        .group_by(["customer_id", "category"])
        .agg(pl.len().alias("category_counts"))
    )

    # 8. Feature: target_user_group_counts
    target_user_group_counts = (
        hist_enriched
        .group_by(["customer_id", "target_user_group_final"])
        .agg(pl.len().alias("target_user_group_counts"))
    )

    # 9. Feature 4: segment_counts theo (customer, category_l1, segment_name)
    segment_counts = (
        hist_enriched
        .group_by(["customer_id", "category_l1", "segment_name"])
        .agg(pl.len().alias("segment_counts"))
    )

    # 10. Feature: time_since_last_purchase_in_B_category
    #     - lấy lần mua gần nhất theo (customer_id, category) trong HIST
    last_cat_purchase = (
        hist_enriched
        .group_by(["customer_id", "category"])
        .agg(
            pl.col("created_date").max().alias("last_purchase_date")
        )
    )

    # 11. Xây tập candidate (customer_id, item_id) từ HIST ∪ RECENT
    hist_pairs = hist_lf.select(["customer_id", "item_id"]).unique()
    recent_pairs = recent_lf.select(["customer_id", "item_id"]).unique()

    candidate_pairs = pl.concat([hist_pairs, recent_pairs]).unique()

    # 12. Enrich candidate với thuộc tính item & segment_name đại diện
    candidate_enriched = (
        candidate_pairs
        .join(item_attrs,   on="item_id", how="left")
        .join(item_segment, on="item_id", how="left")
    )

    # 13. Join tất cả các bảng count + last_cat_purchase để tạo feature set
    features = (
        candidate_enriched
        # brand_counts
        .join(
            brand_counts,
            on=["customer_id", "brand_final"],
            how="left",
        )
        # age_counts
        .join(
            age_counts,
            on=["customer_id", "age_bucket_final"],
            how="left",
        )
        # category_counts
        .join(
            category_counts,
            on=["customer_id", "category"],
            how="left",
        )
        # segment_counts — join theo (customer_id, category_l1, segment_name)
        .join(
            segment_counts,
            on=["customer_id", "category_l1", "segment_name"],
            how="left",
        )
        # target_user_group_counts
        .join(
            target_user_group_counts,
            on=["customer_id", "target_user_group_final"],
            how="left",
        )
        # last_cat_purchase để tính time_since_last_purchase_in_B_category
        .join(
            last_cat_purchase,
            on=["customer_id", "category"],
            how="left",
        )
        # fill null = 0 cho các count
        .with_columns([
            pl.col("brand_counts").fill_null(0),
            pl.col("age_counts").fill_null(0),
            pl.col("category_counts").fill_null(0),
            pl.col("segment_counts").fill_null(0),
            pl.col("target_user_group_counts").fill_null(0),
        ])
        # tính số ngày từ end_hist đến lần mua gần nhất trong cùng category
        .with_columns(
            (
                (pl.lit(end_hist) - pl.col("last_purchase_date"))
                .dt.total_days()
            ).alias("time_since_last_purchase_in_B_category")
        )
        # khách chưa từng mua category đó → 9999
        .with_columns(
            pl.col("time_since_last_purchase_in_B_category").fill_null(9999)
        )
    )

    # 14. Tạo label Y từ RECENT: (customer_id, item_id) có giao dịch trong RECENT -> Y=1
    labels = recent_pairs.with_columns(
        pl.lit(1).alias("Y")
    )

    feature_label_lf = (
        features
        .join(labels, on=["customer_id", "item_id"], how="left")
        .with_columns(
            pl.col("Y").fill_null(0).cast(pl.Int8)
        )
        .select([
            "customer_id",
            "item_id",
            "brand_counts",
            "age_counts",
            "category_counts",
            "segment_counts",
            "target_user_group_counts",
            "time_since_last_purchase_in_B_category",
            "Y",
        ])
    )

    return feature_label_lf


In [16]:
from datetime import datetime

begin_hist = datetime(2024, 1, 1)
end_hist = datetime(2024, 10, 31)   # 30/09/2024, không phải 31/09
begin_recent = datetime(2024, 11, 1)
end_recent = datetime(2024, 11, 30)

feature_label_lf = build_feature_label(
    transactions_lf=purchase_df.lazy(),   # convert DataFrame -> LazyFrame
    items_lf=item_df.lazy(),
    users_lf=user_df.lazy(),
    begin_hist=begin_hist,
    end_hist=end_hist,
    begin_recent=begin_recent,
    end_recent=end_recent,
)

feature_label_df = feature_label_lf.collect()
feature_label_df.head()


/tmp/ipykernel_626286/3273515021.py:29: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  tx_cols = transactions_lf.columns


customer_id,item_id,brand_counts,age_counts,category_counts,segment_counts,target_user_group_counts,time_since_last_purchase_in_B_category,Y
i32,str,u32,u32,u32,u32,u32,i64,i8
7309461,"""1371000000005""",7,23,7,0,11,191,0
4454023,"""7162000000003""",0,5,0,1,5,9999,1
7561357,"""4783000000012""",0,2,0,0,4,9999,1
2048805,"""4372000000001""",2,15,2,60,33,274,0
1641616,"""5503000000004""",2,57,2,0,51,134,0


In [17]:
feature_label_df

customer_id,item_id,brand_counts,age_counts,category_counts,segment_counts,target_user_group_counts,time_since_last_purchase_in_B_category,Y
i32,str,u32,u32,u32,u32,u32,i64,i8
7309461,"""1371000000005""",7,23,7,0,11,191,0
4454023,"""7162000000003""",0,5,0,1,5,9999,1
7561357,"""4783000000012""",0,2,0,0,4,9999,1
2048805,"""4372000000001""",2,15,2,60,33,274,0
1641616,"""5503000000004""",2,57,2,0,51,134,0
…,…,…,…,…,…,…,…,…
5507421,"""2044000000199""",0,10,0,1,7,9999,1
4266894,"""0020010000438""",1,28,1,2,19,163,0
6301728,"""5537000000003""",5,25,5,0,53,17,0


In [18]:
feature_label_df['Y'].value_counts()

Y,count
i8,u32
1,2659290
0,19987486
